# 性能分析与优化

学习目标：为固定工作负载建立计时、调用剖析和内存观察，并在保持结果正确的前提下比较一次数据结构优化的收益与成本。

前置知识：列表与集合、成员检测、函数与类型标注、异常处理、with、对象引用与垃圾回收。

运行环境：Python 3.12。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

示例使用固定小数据，计时通常需要几秒；每次观察后关闭 tracemalloc。

本章保存的时间和内存数值来自实际执行。机器负载、解释器构建及其他并行任务都会影响观察，不能将本章数值推广为一般性能保证。

配套脚本：位于 [scripts/29-performance-analysis/](scripts/29-performance-analysis/)。

（1）[profile\_selection.py](scripts/29-performance-analysis/profile_selection.py)：在独立 Python 进程中剖析本章固定筛选负载，打印调用报告，不生成额外文件。

## 1 先定义工作负载与正确结果

优化前先明确程序要做什么、输入规模有多大，以及测量包含哪些步骤。本章从请求编号中筛出允许访问的编号，保留请求原顺序和重复次数，不修改输入。输入约定为整数列表。

基线使用列表成员检测。CPython 3.12 的列表查询沿元素比较，遇到匹配才停止；若多数查询未命中，就可能重复扫描整个允许列表。设 n 为请求数、m 为允许编号数，最坏情形的比较次数随 n×m 增长；两者同步增长时呈二次增长。这是按查询算法分析比较次数，不是用一次耗时证明复杂度。

固定工作负载在计时前创建：1600 个请求，400 个允许编号。测量筛选函数的一次完整调用，包含结果列表构造，不包含输入准备与打印。

In [1]:
def select_with_list(
    request_ids: list[int], allowed_ids: list[int]
) -> list[int]:
    """筛选允许的整数编号，保留请求顺序及重复次数。"""
    return [
        request_id for request_id in request_ids if request_id in allowed_ids
    ]


request_ids = list(range(800)) * 2
allowed_ids = list(range(400))
expected_ids = list(range(400)) * 2
baseline_ids = select_with_list(request_ids, allowed_ids)

assert baseline_ids == expected_ids
print(len(request_ids), len(allowed_ids), len(baseline_ids))  # 1600 400 800
print(baseline_ids[:4], baseline_ids[-4:])
# [0, 1, 2, 3] [396, 397, 398, 399]：请求中的两轮命中都保留。

1600 400 800
[0, 1, 2, 3] [396, 397, 398, 399]


## 2 分清经过时间与 CPU 时间

经过时间（wall-clock time）回答“从开始到结束等了多久”，其中包含等待；CPU 时间回答当前进程实际使用了多少处理器时间，不包含睡眠。两种计时器都应比较前后差值，不把单次读数解释为日期时间。

| API | 中文名称／含义 |
| --- | --- |
| time.perf\_counter | 高分辨率经过时间，包含睡眠等等待 |
| time.process\_time | 当前进程的用户态与系统态 CPU 时间总和，不包含睡眠 |

下面用短暂睡眠帮助区分口径，不把它当作筛选算法的性能测试。process\_time 是整个进程的口径；Notebook 内核其他线程的 CPU 活动也可能包含在差值中。

In [2]:
import time

wall_start = time.perf_counter()
cpu_start = time.process_time()
time.sleep(0.02)
wall_seconds = time.perf_counter() - wall_start
cpu_seconds = time.process_time() - cpu_start

print(f"经过时间：{wall_seconds:.6f} 秒")
print(f"进程 CPU 时间：{cpu_seconds:.6f} 秒")
# 通常观察到等待主要计入经过时间；调度可能让睡眠超出请求时长。
# 不断言两者的固定比值；计时器分辨率也可能使很短的 CPU 时间显示为零。

经过时间：0.020442 秒
进程 CPU 时间：0.000000 秒


## 3 用 timeit 重复测量小片段

### 3.1 预热、重复次数与每次调用耗时

timeit 默认使用 perf\_counter，适合小片段计时。Timer 的 stmt 是被测语句，globals 提供该语句使用的名称；setup 若有准备步骤，其执行时间不计入被测时间。

repeat=3 表示做三轮测量，number=5 表示每轮调用五次；返回的每个数是该轮总秒数。每次调用的耗时要用轮总时间除以 number，乘 1 000 000 可换成微秒。

下面先执行少量预热，使观察侧重已经运行过的路径；这不测量首次导入或冷启动。查看完整重复结果，再用最小值作为本机这些试次中干扰较少的观察，不把它解释成用户平均响应时间。

In [3]:
import timeit

for _ in range(3):
    select_with_list(request_ids, allowed_ids)

baseline_timer = timeit.Timer(
    "select_with_list(request_ids, allowed_ids)",
    globals={
        "select_with_list": select_with_list,
        "request_ids": request_ids,
        "allowed_ids": allowed_ids,
    },
)
initial_totals = baseline_timer.repeat(repeat=3, number=5)
initial_us = [seconds / 5 * 1_000_000 for seconds in initial_totals]
print("各轮微秒/次：", [round(value, 2) for value in initial_us])
print(f"最小值：{min(initial_us):.2f} 微秒/次")
# 输出来自实际计时；不设置“必须低于多少微秒”的通过阈值。

各轮微秒/次： [5232.32, 4724.16, 4994.16]
最小值：4724.16 微秒/次


### 3.2 timeit 默认临时关闭自动垃圾回收

timeit 默认在计时期间关闭自动循环垃圾回收，以减少不同试次之间的差异；这不关闭 CPython 的引用计数。如果实际工作涉及大量循环垃圾，应通过 setup 重新启用 gc，并明确报告测量口径。

下面只记录计时语句运行时的开关状态，观察默认设置与 setup="gc.enable()" 的区别；这段诊断本身不用于比较性能。finally 恢复进入前的状态，避免影响后续实验。

In [4]:
import gc

gc_states = []
gc_was_enabled = gc.isenabled()
gc_namespace = {"gc": gc, "gc_states": gc_states}
try:
    timeit.Timer(
        "gc_states.append(gc.isenabled())", globals=gc_namespace
    ).timeit(number=1)
    timeit.Timer(
        "gc_states.append(gc.isenabled())",
        setup="gc.enable()",
        globals=gc_namespace,
    ).timeit(number=1)
finally:
    if gc_was_enabled:
        gc.enable()
    else:
        gc.disable()

print(gc_states)  # [False, True]：默认关闭，setup 可重新开启。
print(gc.isenabled() == gc_was_enabled)  # True：恢复进入前的状态。

[False, True]
True


## 4 用 cProfile 定位调用成本

cProfile 记录调用次数和时间，pstats 整理这些统计。它适合定位时间花在哪条调用路径；剖析自身会增加开销，不能直接把剖析报告当作优化前后的基准。

| 报告字段 | 中文名称／含义 |
| --- | --- |
| ncalls | 调用次数；递归时也会显示原始调用次数 |
| tottime | 函数自身的时间，不含子函数调用时间 |
| cumtime | 从调用到返回的累计时间，包含子函数 |
| percall | 左侧时间除以相应调用次数；报告中出现两列 |

下面显式选择 perf\_counter，所以统计口径是经过时间，并非 CPU 使用率。SortKey.CUMULATIVE 按累计时间排序，帮助找到耗时路径；可改用 SortKey.TIME 观察函数自身成本。短任务显示 0.000 可能只是四舍五入，不代表没有成本。

配套脚本用与前文相同的筛选逻辑和输入，在独立进程中重复三批。这样能把报告范围限定在脚本调用中。Notebook 用当前解释器启动脚本；subprocess.run 等待结束，非零退出状态会通过 check=True 报错，超时会结束并等待子进程。

In [5]:
import os
from pathlib import Path
import subprocess
import sys

profile_script = Path("scripts/29-performance-analysis/profile_selection.py")
profile_env = os.environ.copy()
profile_env["PYTHONDONTWRITEBYTECODE"] = "1"
profile_run = subprocess.run(
    [sys.executable, "-B", "-X", "utf8", str(profile_script)],
    cwd=Path.cwd(),
    env=profile_env,
    check=True,
    capture_output=True,
    text=True,
    encoding="utf-8",
    timeout=20,
)
print(profile_run.stdout, end="")
if profile_run.stderr:
    print(profile_run.stderr, end="")
assert profile_run.stdout.splitlines()[0] == "选中总数：2400"
# count_selected_batches 调用一次，select_with_list 调用三次。
# Python 3.12 将本例列表推导式内联，不再显示独立的推导式调用行。
# 整个进程的启动时间没有被脚本内的 cProfile 区间包含。

选中总数：2400
         8 function calls in 0.014 seconds

   Ordered by: cumulative time

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.000    0.000    0.014    0.014 profile_selection.py:17(count_selected_batches)
        3    0.014    0.005    0.014    0.005 profile_selection.py:8(select_with_list)
        1    0.000    0.000    0.000    0.000 {method 'disable' of '_lsprof.Profiler' objects}
        3    0.000    0.000    0.000    0.000 {built-in method builtins.len}




## 5 优化查找结构并保持行为

### 5.1 用集合辅助成员检测

集合的成员查询使用哈希查找，可避免每个请求都从允许列表开头逐项扫描；哈希冲突仍可能增加查询工作，不能把所有输入都说成同样快。

本例只把允许列表转成集合，仍按原请求列表遍历，因此保留请求顺序与重复次数。集合要求元素可哈希，不能把这个优化不加判断地推广到包含列表等不可哈希元素的输入。

集合本身需要构造时间和额外内存。新函数每次调用都构造集合，后面的计时也包含这一步；如果业务复用一个预建集合，应另外定义缓存更新规则与测量边界。

In [6]:
def select_with_set(
    request_ids: list[int], allowed_ids: list[int]
) -> list[int]:
    """用集合辅助筛选整数编号，保留请求顺序及重复次数。"""
    allowed_lookup = set(allowed_ids)
    return [
        request_id for request_id in request_ids if request_id in allowed_lookup
    ]


optimized_ids = select_with_set(request_ids, allowed_ids)
assert optimized_ids == expected_ids
print(optimized_ids == baseline_ids)  # True：优化前后结果逐项相等。
# 没有把 request_ids 转成集合，否则会丢失原请求的顺序与重复次数。

True


### 5.2 先检查边界，再比较速度

仅检查结果长度不足以证明等价：排序变化、重复项丢失也可能保留长度。这里检查完整列表、空输入、重复允许项、负数与未命中编号，并检查输入没有被修改。

这些断言核对本例的业务约定，不要求优化版在每种小输入上都更快；可读性、内存成本和适用条件仍要一起考虑。

In [7]:
edge_cases = [
    ([], [1], []),
    ([1, 2], [], []),
    ([3, 1, 3, 2], [3, 3, 1], [3, 1, 3]),
    ([-1, 0, -1, 9], [0, -1], [-1, 0, -1]),
    ([7, 8], [1, 2], []),
]
for requests, allowed, expected in edge_cases:
    saved_requests = requests.copy()
    saved_allowed = allowed.copy()
    assert select_with_list(requests, allowed) == expected
    assert select_with_set(requests, allowed) == expected
    assert requests == saved_requests
    assert allowed == saved_allowed

print("边界用例通过：", len(edge_cases))  # 5：两种实现均满足同一约定。

边界用例通过： 5


## 6 在相同边界下比较收益

沿用 1600 个请求、400 个允许编号，分别预热两种函数，再交替收集五轮局部对照，每轮调用十次。输入准备和打印均在计时之外，输出列表和集合构造在计时之内，两者都采用 timeit 默认的 gc 口径。

比较值为“基线最小每次耗时 ÷ 优化版最小每次耗时”。这个比值只描述本次工作负载与测量条件；不同命中率、数据量和机器负载可能改变收益。两列完整结果保留，用于观察波动，不设置固定提速倍数断言。

In [8]:
optimized_timer = timeit.Timer(
    "select_with_set(request_ids, allowed_ids)",
    globals={
        "select_with_set": select_with_set,
        "request_ids": request_ids,
        "allowed_ids": allowed_ids,
    },
)
for _ in range(3):
    select_with_list(request_ids, allowed_ids)
    select_with_set(request_ids, allowed_ids)

baseline_totals = []
optimized_totals = []
for _ in range(5):
    baseline_totals.append(baseline_timer.timeit(number=10))
    optimized_totals.append(optimized_timer.timeit(number=10))

baseline_us = [seconds / 10 * 1_000_000 for seconds in baseline_totals]
optimized_us = [seconds / 10 * 1_000_000 for seconds in optimized_totals]
print("列表各轮微秒/次：", [round(value, 2) for value in baseline_us])
print("集合各轮微秒/次：", [round(value, 2) for value in optimized_us])
print(f"列表最小值：{min(baseline_us):.2f} 微秒/次")
print(f"集合最小值：{min(optimized_us):.2f} 微秒/次")
print(f"本次最小值之比：{min(baseline_us) / min(optimized_us):.2f}")
# 先前的正确性检查独立于这些计时数值；比值不构成通用加速保证。

列表各轮微秒/次： [5196.98, 5496.89, 5464.39, 6627.57, 5185.28]
集合各轮微秒/次： [65.34, 64.61, 63.63, 75.04, 68.8]
列表最小值：5185.28 微秒/次
集合最小值：63.63 微秒/次
本次最小值之比：81.49


## 7 用 tracemalloc 观察分配成本

### 7.1 当前分配、历史峰值与重置

tracemalloc 跟踪 Python 分配器中的内存块，不等于进程驻留内存（resident set size，RSS）。它不能自动覆盖所有原生库的分配，也不包括启动跟踪前已经分配的块；结果不是整个进程占用了多少物理内存。

| API | 中文名称／含义 |
| --- | --- |
| tracemalloc.start | 启动内存分配跟踪 |
| tracemalloc.get\_traced\_memory | 返回当前被跟踪字节数与记录的峰值字节数 |
| tracemalloc.reset\_peak | 把峰值重置为当前值，不释放活对象，也不清除分配记录 |
| tracemalloc.stop | 停止跟踪并清除跟踪记录 |

下面先分配一个仍需保留的缓冲区，再创建和释放一个更大的临时缓冲区。重置后仍保留的分配不会变成零。函数在 finally 中关闭自己开启的跟踪；若已经存在其他跟踪会话，就明确报错，避免清除别人的记录。

In [9]:
import tracemalloc


def observe_peak_reset() -> tuple[tuple[int, int], tuple[int, int], int]:
    """观察释放临时分配后，重置峰值仍保留活对象的计数。"""
    if tracemalloc.is_tracing():
        raise RuntimeError("已有内存跟踪会话，请结束该会话后再运行")
    tracemalloc.start()
    try:
        retained = bytearray(4096)
        transient = bytearray(32768)
        del transient
        before_reset = tracemalloc.get_traced_memory()
        tracemalloc.reset_peak()
        after_reset = tracemalloc.get_traced_memory()
        return before_reset, after_reset, len(retained)
    finally:
        tracemalloc.stop()


before_reset, after_reset, retained_size = observe_peak_reset()
print("重置前：当前字节、峰值字节", before_reset)
print("重置后：当前字节、峰值字节", after_reset)
print("仍保留的缓冲区字节数：", retained_size)  # 4096：重置不释放它。
# 读取结果本身也可能产生少量分配，不断言两个时刻的当前值完全相等。
# 重置不清零活分配；数值含对象开销，不应恰好等于缓冲区长度。

重置前：当前字节、峰值字节 (4153, 36978)
重置后：当前字节、峰值字节 (4209, 4209)
仍保留的缓冲区字节数： 4096


### 7.2 比较完整调用的当前增量与峰值增量

下面分别为两种筛选函数启动新的跟踪区间。输入在跟踪开始前已经存在，测量包括函数返回的结果列表；读取计数时仍保留结果，保证两种实现的观察条件一致。

当前增量是读取时的当前字节数减去基线；峰值增量是该区间的峰值减去基线。局部集合在函数返回后不再需要，当前增量可能主要剩下结果列表，峰值才有机会反映临时查找结构的成本。

时间比较已经在关闭 tracemalloc 时完成。跟踪器本身有开销，内存观察期间的运行时间不能直接与前面的基准混用；峰值也只是这次受跟踪分配的观察。

In [10]:
from collections.abc import Callable


def measure_selection_memory(
    selector: Callable[[list[int], list[int]], list[int]],
    request_ids: list[int],
    allowed_ids: list[int],
) -> tuple[int, int, int]:
    """在保留返回列表时测量当前与峰值字节增量，并返回结果项数。"""
    if tracemalloc.is_tracing():
        raise RuntimeError("已有内存跟踪会话，请结束该会话后再运行")
    tracemalloc.start()
    try:
        baseline, _ = tracemalloc.get_traced_memory()
        tracemalloc.reset_peak()
        selected = selector(request_ids, allowed_ids)
        current, peak = tracemalloc.get_traced_memory()
        return current - baseline, peak - baseline, len(selected)
    finally:
        tracemalloc.stop()


for label, selector in (
    ("列表", select_with_list),
    ("集合", select_with_set),
):
    current_bytes, peak_bytes, count = measure_selection_memory(
        selector, request_ids, allowed_ids
    )
    assert count == 800
    print(label, "当前增量字节：", current_bytes, "峰值增量字节：", peak_bytes)
# 比较峰值以观察临时集合的代价；不把这些字节数当作进程 RSS。
# 输入已在跟踪前创建，两次测量都不包含输入列表原有分配。

列表 当前增量字节： 6880 峰值增量字节： 6928
集合 当前增量字节： 6880 峰值增量字节： 41224


## 本章小结

（1）先确定输入、完整输出约定与测量边界。优化后检查顺序、重复项、边界值和输入是否被修改。

（2）perf\_counter 测量经过时间，process\_time 测量进程 CPU 时间；timeit 适合重复测量小片段，要注明预热、轮数、每轮次数和 gc 口径。

（3）cProfile 帮助定位调用路径，tottime 与 cumtime 含义不同；剖析结果不替代独立计时。

（4）集合查找减少反复扫描，但构造集合需要时间和内存。仅转换允许列表，才能保留请求顺序与重复次数。

（5）tracemalloc 测量受跟踪分配，当前值与峰值回答不同问题；reset\_peak 不释放对象，测量结果也不等于 RSS。

自查：能否给出一次优化的正确性证据、计时边界和内存成本，同时说明这些结论适用于哪些输入？

## 练习

（1）先预测下面两个结果列表，再运行核对。判断把请求与允许编号直接求集合交集是否满足本章约定；核对标准是能指出顺序、重复次数和返回类型三个方面的区别。

In [11]:
exercise_requests = [4, 1, 4, 2]
exercise_allowed = [4, 4, 1]
print(select_with_list(exercise_requests, exercise_allowed))
print(select_with_set(exercise_requests, exercise_allowed))
# 先记录预测，再核对每个位置；不要只比较结果长度。

[4, 1, 4]
[4, 1, 4]


（2）比较“小允许列表”和“全部未命中”两种工作负载。每种输入先检查两种实现的完整结果相等、原输入未修改，再分别预热，做三轮、每轮五次的 timeit 测量。

输出各轮微秒/次与最小值，说明输入准备、集合构造和 gc 分别如何处理。检查轮数与单位换算，不设置集合版必须更快的条件，也不把两种负载的结果混成同一个加速比。

In [12]:
exercise_workloads = [
    (list(range(40)) * 2, [1, 3]),
    (list(range(200, 400)), list(range(100))),
]
# 在此逐个验证结果，再分别建立 Timer；不要在计时语句中打印。
# 三轮结果都应保留，总秒数除以五得到每次调用的秒数。

（3）复用 measure\_selection\_memory 测量下面两种函数，先独立核对它们返回相同的空列表。解释为什么当前增量不能代表全部分配历史，且一次峰值观察不能判断程序存在内存泄漏。

检查跟踪结束后 is\_tracing 为 False、峰值不小于同次当前增量；报告实际字节数，不断言某个精确差值或固定倍数。临时缓冲区在返回前不再被引用，输入依旧在跟踪前创建。

In [13]:
def empty_selection(
    request_ids: list[int], allowed_ids: list[int]
) -> list[int]:
    """为内存练习返回空结果。"""
    return []


def empty_after_buffer(
    request_ids: list[int], allowed_ids: list[int]
) -> list[int]:
    """分配并释放一个临时缓冲区后返回空结果。"""
    buffer = bytearray(16384)
    del buffer
    return []


# 在此先检查两种函数的返回值，再分别测量当前与峰值增量。
# 此处只控制临时分配差异，不用运行时间比较这两个练习函数。

## 参考与引用来源

| 网站 | 本章参考内容与定位 |
| --- | --- |
| Python 官方文档（3.12） | [集合语义与可哈希条件](https://docs.python.org/3.12/library/stdtypes.html#set-types-set-frozenset)、[数据模型中的集合用途](https://docs.python.org/3.12/reference/datamodel.html#set-types)；[经过时间](https://docs.python.org/3.12/library/time.html#time.perf_counter)、[进程 CPU 时间](https://docs.python.org/3.12/library/time.html#time.process_time)、[睡眠与调度](https://docs.python.org/3.12/library/time.html#time.sleep)；[timeit 默认计时器](https://docs.python.org/3.12/library/timeit.html#timeit.default_timer)、[Timer、命名空间与 setup 边界](https://docs.python.org/3.12/library/timeit.html#timeit.Timer)、[每轮次数与 gc 行为](https://docs.python.org/3.12/library/timeit.html#timeit.Timer.timeit)、[repeat 与最小值的解释](https://docs.python.org/3.12/library/timeit.html#timeit.Timer.repeat)、[gc 开关与引用计数](https://docs.python.org/3.12/library/gc.html)；[剖析工具用途与基准限制](https://docs.python.org/3.12/library/profile.html#introduction-to-the-profilers)、[报告列与读取方式](https://docs.python.org/3.12/library/profile.html#instant-user-s-manual)、[排序依据](https://docs.python.org/3.12/library/profile.html#pstats.Stats.sort_stats)、[Profile 与 runcall](https://docs.python.org/3.12/library/profile.html#profile.Profile)、[自定义计时器](https://docs.python.org/3.12/library/profile.html#using-a-custom-timer)、[子进程等待、退出状态、超时与环境](https://docs.python.org/3.12/library/subprocess.html#subprocess.run)、[3.12 推导式内联](https://docs.python.org/3.12/whatsnew/3.12.html#pep-709-comprehension-inlining)；[tracemalloc 范围与分配跟踪](https://docs.python.org/3.12/library/tracemalloc.html)、[开始跟踪与额外开销](https://docs.python.org/3.12/library/tracemalloc.html#tracemalloc.start)、[当前值与峰值](https://docs.python.org/3.12/library/tracemalloc.html#tracemalloc.get_traced_memory)、[峰值重置不清除记录](https://docs.python.org/3.12/library/tracemalloc.html#tracemalloc.reset_peak)、[停止与清理](https://docs.python.org/3.12/library/tracemalloc.html#tracemalloc.stop)、[不包含跟踪前分配](https://docs.python.org/3.12/library/tracemalloc.html#tracemalloc.take_snapshot)。 |
| GitHub：CPython 官方源码（v3.12.14） | [Objects/listobject.c：list\_contains，逐项比较与提前停止](https://github.com/python/cpython/blob/v3.12.14/Objects/listobject.c#L441-L454)；[Objects/setobject.c：set\_lookkey，哈希定位与冲突探查](https://github.com/python/cpython/blob/v3.12.14/Objects/setobject.c#L56-L105)、[set\_contains\_key，计算哈希并查找](https://github.com/python/cpython/blob/v3.12.14/Objects/setobject.c#L358-L369)。 |